# Phase 1 — Baseline Evaluation

Local holdout for **Baseline A** (Dragapult only) vs **Baseline B** (Dragapult + UCB1 search).

Opponent panel (research plan): Alakazam, Crustle, Spidops, Starmie.

Kaggle tasks (ladder submissions) are documented in `docs/PHASE_01_LOG.md`.


In [ ]:
!pip3 install -r ../requirements.txt

  Using cached kaggle_environments-1.32.2-py3-none-any.whl.metadata (828 bytes)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached gymnasium-1.2.0-py3-none-any.whl.metadata (9.9 kB)
  Using cached gymnax-0.0.8-py3-none-any.whl.metadata (19 kB)
  Using cached jax-0.10.2-py3-none-any.whl.metadata (13 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 3.4 MB/s  0:00:05 eta 0:00:01
  Installing build dependencies ... 

In [1]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

NOTEBOOKS = Path.cwd() if (Path.cwd() / "env_paths.py").exists() else Path.cwd() / "notebooks"
if str(NOTEBOOKS) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS))

from env_paths import get_paths, setup_runtime

PATHS = get_paths()
setup_runtime(PATHS)

# Subprocess cells must use the project .venv (system Python lacks kaggle-environments).
VENV_PYTHON = PATHS.repo_root / ".venv" / "bin" / "python"
PYTHON = str(VENV_PYTHON if VENV_PYTHON.exists() else sys.executable)

print("repo_root:", PATHS.repo_root)
print("python:", PYTHON)
if VENV_PYTHON.exists() and Path(sys.executable).resolve() != VENV_PYTHON.resolve():
    print("Tip: select the project .venv kernel so notebook and subprocess use the same interpreter.")


ModuleNotFoundError: No module named 'pandas'

## 1. Build baseline variants


In [ ]:
builder = NOTEBOOKS / "build_merged_agent.py"
subprocess.run([PYTHON, str(builder), "--variant", "all"], check=True, cwd=str(NOTEBOOKS))


Wrote /Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/agents/main_baseline_a.py
Wrote /Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/agents/main_baseline_b.py
Wrote /Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/merged_agent_main.py
Synced /Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/main.py


CompletedProcess(args=['/usr/local/bin/python', '/Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/build_merged_agent.py', '--variant', 'all'], returncode=0)

## 2. Refresh holdout opponent panel


In [ ]:
extractor = NOTEBOOKS / "extract_holdout_panel.py"
subprocess.run([PYTHON, str(extractor)], check=True, cwd=str(NOTEBOOKS))


Wrote holdout panel under /Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/holdout/panel


CompletedProcess(args=['/usr/local/bin/python', '/Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/extract_holdout_panel.py'], returncode=0)

## 3. Run holdout suite (40 games × 4 opponents × 2 baselines)


In [ ]:
HOLDOUT_GAMES = 40
runner = NOTEBOOKS / "run_phase1_holdout.py"
subprocess.run(
    [PYTHON, str(runner), "--games", str(HOLDOUT_GAMES)],
    check=True,
    cwd=str(NOTEBOOKS),
)


Traceback (most recent call last):
  File "/Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/run_phase1_holdout.py", line 17, in <module>
    from holdout_runner import (
  File "/Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/holdout_runner.py", line 15, in <module>
    from kaggle_environments import make
ModuleNotFoundError: No module named 'kaggle_environments'


CalledProcessError: Command '['/usr/local/bin/python', '/Volumes/Sandisk 2TB/Documents/Hackathons and Competitions/Kaggle - PTCG/notebooks/run_phase1_holdout.py', '--games', '40']' returned non-zero exit status 1.

## 4. Summaries


In [ ]:
import json

summary_path = NOTEBOOKS / "output/phase1/phase1_holdout_summary_latest.json"
summaries = json.loads(summary_path.read_text(encoding="utf-8"))
df = pd.DataFrame(summaries)
display(df[["baseline", "opponent", "wins", "losses", "ties", "win_rate", "holdout_gate"]])


## 5. Kaggle (manual)

See `docs/PHASE_01_LOG.md` — submit Baseline A and Baseline B once each and record ladder ratings.
